# 0. Read the file
If the file location is different, only modify the two paths below. This cell reads the required columns at once, avoiding the need to re-scan the large Stata file for each extracted feature.

In [ ]:
from pathlib import Path
import pandas as pd

folder = Path("/Users/densoul/Desktop/600")
rand_file = folder / "randhrs1992_2022v1_STATA" / "randhrs1992_2022v1.dta"
cog_file = folder / "CogImp2022" / "cogimp9222a_r.dta"

for file in (rand_file, cog_file):
    if not file.is_file():
        raise FileNotFoundError(f"File not found:{file} Please modify the above path.")

waves = {13: 2016, 14: 2018, 15: 2020, 16: 2022}
disease_suffixes = ("hibpe", "diabe", "cancre", "lunge", "hearte", "stroke", "arthre")

rand_cols = ["hhid", "pn", "ragender", "raracem", "rahispan", "raedyrs"]
for w in waves:
    rand_cols += [f"r{w}agey_b", f"r{w}mstat", f"r{w}proxy", f"r{w}smoken", f"r{w}bmi"]
    rand_cols += [f"r{w}{suffix}" for suffix in disease_suffixes]

cog_cols = ["hhid", "pn"]
for w in waves:
    cog_cols.append(f"R{w}COG27")
    if w == 13:
        cog_cols += ["R13IMRC", "R13DLRC", "R13SER7", "R13BWC20"]
    else:
        for suffix in ("IMRC", "DLRC", "SER7", "BWC20"):
            cog_cols += [f"R{w}{suffix}P", f"R{w}{suffix}W"]
        cog_cols.append(f"R{w}COGMODE")

rand = pd.read_stata(rand_file, columns=rand_cols, convert_categoricals=False)
cog = pd.read_stata(cog_file, columns=cog_cols, convert_categoricals=False)

def normalize_id(series, width):
    return (series.astype("string").str.strip()
            .str.replace(r"\.0$", "", regex=True).str.zfill(width))

for frame in (rand, cog):
    frame["hhid"] = normalize_id(frame["hhid"], 6)
    frame["pn"] = normalize_id(frame["pn"], 3)

if rand[["hhid", "pn"]].isna().any().any() or cog[["hhid", "pn"]].isna().any().any():
    raise ValueError("hhid or pn have missing")

merged = rand.merge(cog, on=["hhid", "pn"], how="left",
                    validate="one_to_one", indicator=True)
print("Number of RAND participants:", len(rand), ";CogImp Number of people:", len(cog))
print("The number of people matching the two files:", int(merged["_merge"].eq("both").sum()))

Number of RAND participants: 45234 ；CogImp Number of people: 43801
The number of people matching the two files: 43793


# 1. Create a table with one row per person
First, establish the ID and the year in this column. Then, each variable is placed in the same table in the order of 2016, 2018, 2020, 2022.

In [3]:
analysis = pd.concat(
    [merged[["hhid", "pn"]].assign(**{"Survey year": year})
     for year in waves.values()],
    ignore_index=True
)
analysis.insert(0, "Participant ID", analysis["hhid"] + analysis["pn"])
analysis = analysis.drop(columns=["hhid", "pn"])

def report(name):
    missing_by_year = (
        analysis[name]
        .isna()
        .groupby(analysis["Survey year"])
        .sum()
    )
    print(f"{name} missing by year:", missing_by_year.to_dict())

# 2. extract all the features we need 

## 1. Cognitive Tests and Total Score
Since 2018, CogImp has divided some individual tests into two columns: face-to-face/telephone (P) and web (W). Each cell combines the corresponding two columns into one feature.

## Immediate recall

In [4]:
parts = []
for w in waves:
    if w == 13:
        part = merged["R13IMRC"]
    else:
        phone = merged[f"R{w}IMRCP"]
        web = merged[f"R{w}IMRCW"]
        if (phone.notna() & web.notna()).any():
            raise ValueError("Immediate recall: P/W Both columns have values. Please verify.")
        part = phone.combine_first(web)
    parts.append(part)
analysis["Immediate recall"] = pd.to_numeric(
    pd.concat(parts, ignore_index=True), errors="coerce"
)
report("Immediate recall")

Immediate recall missing by year: {2016: 25263, 2018: 28751, 2020: 30232, 2022: 30041}


## Delayed recall

In [5]:
parts = []
for w in waves:
    if w == 13:
        part = merged["R13DLRC"]
    else:
        phone = merged[f"R{w}DLRCP"]
        web = merged[f"R{w}DLRCW"]
        if (phone.notna() & web.notna()).any():
            raise ValueError("Delayed recall: P/W Both columns have values. Please verify.")
        part = phone.combine_first(web)
    parts.append(part)
analysis["Delayed recall"] = pd.to_numeric(
    pd.concat(parts, ignore_index=True), errors="coerce"
)
report("Delayed recall")

Delayed recall missing by year: {2016: 25263, 2018: 28751, 2020: 30232, 2022: 30041}


## Serial 7s

In [6]:
parts = []
for w in waves:
    if w == 13:
        part = merged["R13SER7"]
    else:
        phone = merged[f"R{w}SER7P"]
        web = merged[f"R{w}SER7W"]
        if (phone.notna() & web.notna()).any():
            raise ValueError("Serial 7s: P/W Both columns have values. Please verify.")
        part = phone.combine_first(web)
    parts.append(part)
analysis["Serial 7s"] = pd.to_numeric(
    pd.concat(parts, ignore_index=True), errors="coerce"
)
report("Serial 7s")

Serial 7s missing by year: {2016: 25263, 2018: 28751, 2020: 30232, 2022: 30041}


## Backwards counting

In [7]:
parts = []
for w in waves:
    if w == 13:
        part = merged["R13BWC20"]
    else:
        phone = merged[f"R{w}BWC20P"]
        web = merged[f"R{w}BWC20W"]
        if (phone.notna() & web.notna()).any():
            raise ValueError("Backwards counting: P/W Both columns have values. Please verify.")
        part = phone.combine_first(web)
    parts.append(part)
analysis["Backwards counting"] = pd.to_numeric(
    pd.concat(parts, ignore_index=True), errors="coerce"
)
report("Backwards counting")

Backwards counting missing by year: {2016: 25263, 2018: 28751, 2020: 30232, 2022: 30041}


## Derived 0–27 cognition score

In [28]:
analysis["Derived 0-27 cognition score"] = pd.to_numeric(
    pd.concat([
    merged["R13COG27"],
    merged["R14COG27"],
    merged["R15COG27"],
    merged["R16COG27"],
], ignore_index=True),
    errors="coerce"
)
report("Derived 0-27 cognition score")

Derived 0-27 cognition score missing by year: {2016: 25263, 2018: 28751, 2020: 30232, 2022: 30041}


# 3. Seven chronic diseases and total number of diseases
Each disease is coded as 0 (absent) or 1 (present); other codes are left as missing values.

## Hypertension

In [9]:
raw = pd.to_numeric(
    pd.concat([
    merged["r13hibpe"],
    merged["r14hibpe"],
    merged["r15hibpe"],
    merged["r16hibpe"],
], ignore_index=True),
    errors="coerce"
)
analysis["Hypertension"] = raw.where(raw.isin([0, 1])).astype("Int8")
report("Hypertension")

Hypertension missing by year: {2016: 24322, 2018: 28088, 2020: 29511, 2022: 29378}


## Diabetes

In [10]:
raw = pd.to_numeric(
    pd.concat([
    merged["r13diabe"],
    merged["r14diabe"],
    merged["r15diabe"],
    merged["r16diabe"],
], ignore_index=True),
    errors="coerce"
)
analysis["Diabetes"] = raw.where(raw.isin([0, 1])).astype("Int8")
report("Diabetes")

Diabetes missing by year: {2016: 24322, 2018: 28088, 2020: 29511, 2022: 29378}


## Cancer

In [11]:
raw = pd.to_numeric(
    pd.concat([
    merged["r13cancre"],
    merged["r14cancre"],
    merged["r15cancre"],
    merged["r16cancre"],
], ignore_index=True),
    errors="coerce"
)
analysis["Cancer"] = raw.where(raw.isin([0, 1])).astype("Int8")
report("Cancer")

Cancer missing by year: {2016: 24322, 2018: 28088, 2020: 29511, 2022: 29378}


## Chronic lung disease

In [12]:
raw = pd.to_numeric(
    pd.concat([
    merged["r13lunge"],
    merged["r14lunge"],
    merged["r15lunge"],
    merged["r16lunge"],
], ignore_index=True),
    errors="coerce"
)
analysis["Chronic lung disease"] = raw.where(raw.isin([0, 1])).astype("Int8")
report("Chronic lung disease")

Chronic lung disease missing by year: {2016: 24322, 2018: 28088, 2020: 29511, 2022: 29378}


## Heart disease

In [13]:
raw = pd.to_numeric(
    pd.concat([
    merged["r13hearte"],
    merged["r14hearte"],
    merged["r15hearte"],
    merged["r16hearte"],
], ignore_index=True),
    errors="coerce"
)
analysis["Heart disease"] = raw.where(raw.isin([0, 1])).astype("Int8")
report("Heart disease")

Heart disease missing by year: {2016: 24322, 2018: 28088, 2020: 29511, 2022: 29378}


## Stroke

In [14]:
raw = pd.to_numeric(
    pd.concat([
    merged["r13stroke"],
    merged["r14stroke"],
    merged["r15stroke"],
    merged["r16stroke"],
], ignore_index=True),
    errors="coerce"
)
analysis["Stroke"] = raw.where(raw.isin([0, 1])).astype("Int8")
report("Stroke")

Stroke missing by year: {2016: 24322, 2018: 28088, 2020: 29511, 2022: 29378}


## Arthritis

In [15]:
raw = pd.to_numeric(
    pd.concat([
    merged["r13arthre"],
    merged["r14arthre"],
    merged["r15arthre"],
    merged["r16arthre"],
], ignore_index=True),
    errors="coerce"
)
analysis["Arthritis"] = raw.where(raw.isin([0, 1])).astype("Int8")
report("Arthritis")

Arthritis missing by year: {2016: 24322, 2018: 28088, 2020: 29511, 2022: 29378}


## Chronic disease count

In [16]:
disease_cols = [
    "Hypertension", "Diabetes", "Cancer", "Chronic lung disease",
    "Heart disease", "Stroke", "Arthritis"
]
analysis["Chronic disease count"] = (
    analysis[disease_cols].sum(axis=1, min_count=7).astype("Int8")
)
report("Chronic disease count")

Chronic disease count missing by year: {2016: 24322, 2018: 28088, 2020: 29511, 2022: 29378}


# 4. Demographic Variables
## Age

In [17]:
analysis["Age"] = pd.to_numeric(
    pd.concat([
    merged["r13agey_b"],
    merged["r14agey_b"],
    merged["r15agey_b"],
    merged["r16agey_b"],
], ignore_index=True),
    errors="coerce"
)
report("Age")

Age missing by year: {2016: 24322, 2018: 28088, 2020: 29511, 2022: 29378}


## Sex

In [18]:
analysis["Sex"] = pd.to_numeric(
    pd.concat([merged["ragender"] for _ in waves], ignore_index=True),
    errors="coerce"
)
report("Sex")

Sex missing by year: {2016: 0, 2018: 0, 2020: 0, 2022: 0}


## Race/ethnicity

In [19]:
race = pd.to_numeric(
    pd.concat([merged["raracem"] for _ in waves], ignore_index=True),
    errors="coerce"
)
hispanic = pd.to_numeric(
    pd.concat([merged["rahispan"] for _ in waves], ignore_index=True),
    errors="coerce"
)
group = pd.Series(pd.NA, index=analysis.index, dtype="string")
group.loc[hispanic.eq(1)] = "Hispanic"
group.loc[hispanic.eq(0) & race.eq(1)] = "Non-Hispanic White"
group.loc[hispanic.eq(0) & race.eq(2)] = "Non-Hispanic Black"
group.loc[hispanic.eq(0) & race.eq(3)] = "Non-Hispanic Other"
analysis["Race/ethnicity"] = group
report("Race/ethnicity")

Race/ethnicity missing by year: {2016: 584, 2018: 584, 2020: 584, 2022: 584}


## Education

In [20]:
analysis["Education"] = pd.to_numeric(
    pd.concat([merged["raedyrs"] for _ in waves], ignore_index=True),
    errors="coerce"
)
report("Education")

Education missing by year: {2016: 649, 2018: 649, 2020: 649, 2022: 649}


## Marital status

In [21]:
analysis["Marital status"] = pd.to_numeric(
    pd.concat([
    merged["r13mstat"],
    merged["r14mstat"],
    merged["r15mstat"],
    merged["r16mstat"],
], ignore_index=True),
    errors="coerce"
)
report("Marital status")

Marital status missing by year: {2016: 24354, 2018: 28114, 2020: 29549, 2022: 29418}


## Smoking

In [22]:
analysis["Smoking"] = pd.to_numeric(
    pd.concat([
    merged["r13smoken"],
    merged["r14smoken"],
    merged["r15smoken"],
    merged["r16smoken"],
], ignore_index=True),
    errors="coerce"
)
report("Smoking")

Smoking missing by year: {2016: 24416, 2018: 28159, 2020: 29683, 2022: 31784}


## BMI

In [26]:
analysis["BMI"] = pd.to_numeric(
    pd.concat([
    merged["r13bmi"],
    merged["r14bmi"],
    merged["r15bmi"],
    merged["r16bmi"],
], ignore_index=True),
    errors="coerce"
)
report("BMI")

BMI missing by year: {2016: 24656, 2018: 28331, 2020: 29737, 2022: 29661}


## Interview mode

In [25]:
analysis["Interview mode"] = pd.to_numeric(
    pd.concat([
        pd.Series(pd.NA, index=merged.index),
        merged["R14COGMODE"],
        merged["R15COGMODE"],
        merged["R16COGMODE"],
    ], ignore_index=True),
    errors="coerce"
)
report("Interview mode")

Interview mode missing by year: {2016: 45234, 2018: 28751, 2020: 30232, 2022: 30041}


## Proxy respondent

In [24]:
analysis["Proxy respondent"] = pd.to_numeric(
    pd.concat([
    merged["r13proxy"],
    merged["r14proxy"],
    merged["r15proxy"],
    merged["r16proxy"],
], ignore_index=True),
    errors="coerce"
)
report("Proxy respondent")

Proxy respondent missing by year: {2016: 24322, 2018: 28088, 2020: 29511, 2022: 29378}


# 5. Check and export CSV
Output two files: The first file retains all records with cognitive total scores; the second file only retains records of individuals whose age was at least 60 years old in that year. The 60-year-old samples may still have missing values for other variables. Later, the Analysis ready (age 60+) tag can be used to select the complete sample of the main model.

In [29]:
ordered_cols = [
    "Participant ID", "Survey year",
    "Immediate recall", "Delayed recall", "Serial 7s",
    "Backwards counting", "Derived 0-27 cognition score",
    "Hypertension", "Diabetes", "Cancer", "Chronic lung disease",
    "Heart disease", "Stroke", "Arthritis", "Chronic disease count",
    "Age", "Sex", "Race/ethnicity", "Education", "Marital status", "Smoking", "BMI",
    "Interview mode", "Proxy respondent"
]

output = analysis[ordered_cols].copy()

if output.duplicated(["Participant ID", "Survey year"]).any():
    raise ValueError("Duplicate participant-year records found.")

csv_file = folder / "hrs_full_data.csv"
output.to_csv(csv_file, index=False, encoding="utf-8-sig")

print(f"Saved: {csv_file}")
print(f"Participant-year records: {len(output):,}")
print("Records by survey year:")
print(output.groupby("Survey year").size().to_string())

Saved: /Users/densoul/Desktop/600/hrs_full_data.csv
Participant-year records: 180,936
Records by survey year:
Survey year
2016    45234
2018    45234
2020    45234
2022    45234


# 6.Check for missing values

In [30]:
feature_cols = [
    column for column in output.columns
    if column not in ["Participant ID", "Survey year"]
]

year_counts = output.groupby("Survey year").size()
missing_counts = (
    output[feature_cols]
    .isna()
    .groupby(output["Survey year"])
    .sum()
    .T
)
missing_percent = (
    missing_counts.div(year_counts, axis=1)
    .mul(100)
    .round(1)
)

print("Missing counts by survey year:")
print(missing_counts.to_string())

print("Missing percentages by survey year:")
print(missing_percent.to_string())

print("Overall missing counts:")
print(
    output[feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

Missing counts by survey year:
Survey year                    2016   2018   2020   2022
Immediate recall              25263  28751  30232  30041
Delayed recall                25263  28751  30232  30041
Serial 7s                     25263  28751  30232  30041
Backwards counting            25263  28751  30232  30041
Derived 0-27 cognition score  25263  28751  30232  30041
Hypertension                  24322  28088  29511  29378
Diabetes                      24322  28088  29511  29378
Cancer                        24322  28088  29511  29378
Chronic lung disease          24322  28088  29511  29378
Heart disease                 24322  28088  29511  29378
Stroke                        24322  28088  29511  29378
Arthritis                     24322  28088  29511  29378
Chronic disease count         24322  28088  29511  29378
Age                           24322  28088  29511  29378
Sex                               0      0      0      0
Race/ethnicity                  584    584    584    584
